# Notebook de benchmark JupyterHub

Ce notebook permet de mesurer ta plateforme de facon distincte et simultanee sur 5 axes :
- CPU
- RAM
- ecriture disque
- lecture disque
- charge combinee CPU + RAM + disque

Chaque execution produit des metriques exploitables :
- temps total
- CPU moyen et pic
- memoire RSS moyenne et pic
- memoire disponible minimale
- debit disque de lecture et d'ecriture quand applicable
- series temporelles exportees en CSV

Les artefacts sont ecrits dans `benchmark_outputs/<run_id>/`.

Note pratique : le test de lecture disque peut etre influence par le cache OS. Pour un test plus severe, augmente la taille du fichier et relance le benchmark quand la machine est peu chargee.

In [ ]:
# Si besoin, decommenter la ligne suivante dans Jupyter :
# %pip install numpy pandas matplotlib psutil

import json
import math
import os
import platform
import socket
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("ggplot")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    "artifact_root": "benchmark_outputs",
    "sample_interval_s": 0.50,
    "random_seed": 42,
    "cleanup_temp_files": True,
    "cpu_matrix_size": 1200,
    "cpu_repeats": 4,
    "memory_array_gb": 0.25,
    "memory_cycles": 6,
    "disk_file_size_mb": 1024,
    "disk_chunk_mb": 16,
    "combined_cpu_matrix_size": 900,
    "combined_cpu_repeats": 3,
    "combined_memory_gb": 0.15,
    "combined_memory_cycles": 4,
}

ARTIFACT_DIR = Path(CONFIG["artifact_root"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

disk_usage = shutil_disk = None
try:
    shutil_disk = psutil.disk_usage(str(ARTIFACT_DIR.resolve().anchor or Path.cwd()))
except Exception:
    shutil_disk = None

ENV_INFO = {
    "run_id": RUN_ID,
    "hostname": socket.gethostname(),
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "logical_cpus": psutil.cpu_count(logical=True),
    "physical_cpus": psutil.cpu_count(logical=False),
    "total_memory_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2),
    "artifact_dir": str(ARTIFACT_DIR.resolve()),
    "disk_free_gb": round(shutil_disk.free / (1024 ** 3), 2) if shutil_disk else None,
}

pd.Series(ENV_INFO, name="value").to_frame()

In [ ]:
def format_bytes(num_bytes):
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(num_bytes)
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024


def normalize_for_json(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {key: normalize_for_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [normalize_for_json(item) for item in value]
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


def persist_json(path, payload):
    path = Path(path)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(normalize_for_json(payload), handle, indent=2)


def safe_unlink(path, retries=10, delay_s=0.2):
    path = Path(path)
    if not path.exists():
        return True
    last_error = None
    for _ in range(retries):
        try:
            path.unlink()
            return True
        except PermissionError as exc:
            last_error = exc
            time.sleep(delay_s)
    if path.exists() and last_error is not None:
        print(f"Warning: impossible de supprimer temporairement {path} ({last_error})")
    return not path.exists()


def write_file_benchmark(target_path, file_size_mb, chunk_mb):
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    total_bytes = int(file_size_mb * 1024 ** 2)
    chunk_bytes = int(chunk_mb * 1024 ** 2)
    block = os.urandom(chunk_bytes)
    remaining = total_bytes
    start = time.perf_counter()
    with target_path.open("wb", buffering=0) as handle:
        while remaining > 0:
            payload = block if remaining >= chunk_bytes else block[:remaining]
            written = handle.write(payload)
            remaining -= written
        handle.flush()
        os.fsync(handle.fileno())
    elapsed = time.perf_counter() - start
    return {
        "disk_write_elapsed_s": elapsed,
        "disk_write_mb_s": total_bytes / elapsed / (1024 ** 2),
        "disk_write_size_mb": file_size_mb,
        "disk_write_path": str(target_path.resolve()),
    }


def ensure_seed_file(seed_path, file_size_mb, chunk_mb):
    seed_path = Path(seed_path)
    expected_size = int(file_size_mb * 1024 ** 2)
    if seed_path.exists() and seed_path.stat().st_size == expected_size:
        return seed_path
    _ = write_file_benchmark(seed_path, file_size_mb=file_size_mb, chunk_mb=chunk_mb)
    return seed_path


def read_file_benchmark(source_path, chunk_mb):
    source_path = Path(source_path)
    chunk_bytes = int(chunk_mb * 1024 ** 2)
    total_bytes = source_path.stat().st_size
    consumed_bytes = 0
    checksum = 0
    start = time.perf_counter()
    with source_path.open("rb", buffering=0) as handle:
        while True:
            payload = handle.read(chunk_bytes)
            if not payload:
                break
            consumed_bytes += len(payload)
            checksum ^= payload[0]
    elapsed = time.perf_counter() - start
    return {
        "disk_read_elapsed_s": elapsed,
        "disk_read_mb_s": consumed_bytes / elapsed / (1024 ** 2),
        "disk_read_size_mb": total_bytes / (1024 ** 2),
        "disk_read_checksum": checksum,
        "disk_read_path": str(source_path.resolve()),
    }

In [ ]:
class MetricSampler:
    def __init__(self, interval_s=0.5):
        self.interval_s = interval_s
        self.process = psutil.Process(os.getpid())
        self.samples = []
        self._stop_event = threading.Event()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._t0 = None

    def capture(self):
        memory = psutil.virtual_memory()
        disk = psutil.disk_io_counters()
        self.samples.append(
            {
                "t_s": time.perf_counter() - self._t0,
                "system_cpu_pct": psutil.cpu_percent(interval=None),
                "process_cpu_pct": self.process.cpu_percent(interval=None),
                "process_rss_gb": self.process.memory_info().rss / (1024 ** 3),
                "available_memory_gb": memory.available / (1024 ** 3),
                "used_memory_pct": memory.percent,
                "system_disk_read_mb": (disk.read_bytes / (1024 ** 2)) if disk else np.nan,
                "system_disk_write_mb": (disk.write_bytes / (1024 ** 2)) if disk else np.nan,
            }
        )

    def _loop(self):
        while not self._stop_event.wait(self.interval_s):
            self.capture()

    def start(self):
        self._t0 = time.perf_counter()
        psutil.cpu_percent(interval=None)
        self.process.cpu_percent(interval=None)
        self.capture()
        self._thread.start()

    def stop(self):
        self.capture()
        self._stop_event.set()
        self._thread.join(timeout=self.interval_s * 2)
        return pd.DataFrame(self.samples)


def summarize_samples(samples_df):
    if samples_df.empty:
        return {}
    summary = {
        "sample_count": int(len(samples_df)),
        "avg_system_cpu_pct": samples_df["system_cpu_pct"].mean(),
        "peak_system_cpu_pct": samples_df["system_cpu_pct"].max(),
        "avg_process_cpu_pct": samples_df["process_cpu_pct"].mean(),
        "peak_process_cpu_pct": samples_df["process_cpu_pct"].max(),
        "avg_process_rss_gb": samples_df["process_rss_gb"].mean(),
        "peak_process_rss_gb": samples_df["process_rss_gb"].max(),
        "min_available_memory_gb": samples_df["available_memory_gb"].min(),
        "max_used_memory_pct": samples_df["used_memory_pct"].max(),
    }
    if samples_df["system_disk_read_mb"].notna().all():
        summary["system_disk_read_delta_mb"] = (
            samples_df["system_disk_read_mb"].iloc[-1] - samples_df["system_disk_read_mb"].iloc[0]
        )
    if samples_df["system_disk_write_mb"].notna().all():
        summary["system_disk_write_delta_mb"] = (
            samples_df["system_disk_write_mb"].iloc[-1] - samples_df["system_disk_write_mb"].iloc[0]
        )
    return summary


def run_with_monitor(test_name, workload, config):
    sampler = MetricSampler(interval_s=config["sample_interval_s"])
    start_ts = datetime.now().isoformat(timespec="seconds")
    sampler.start()
    wall_start = time.perf_counter()
    try:
        workload_metrics = workload(config)
    finally:
        wall_elapsed = time.perf_counter() - wall_start
        samples_df = sampler.stop()

    summary = {
        "test_name": test_name,
        "started_at": start_ts,
        "elapsed_s": wall_elapsed,
    }
    summary.update(summarize_samples(samples_df))
    summary.update(workload_metrics)
    return {
        "summary": summary,
        "samples": samples_df,
    }

In [ ]:
def cpu_kernel(matrix_size, repeats, seed):
    rng = np.random.default_rng(seed)
    a = rng.random((matrix_size, matrix_size), dtype=np.float32)
    b = rng.random((matrix_size, matrix_size), dtype=np.float32)
    checksum = 0.0
    start = time.perf_counter()
    for _ in range(repeats):
        c = a @ b
        checksum += float(c[0, 0])
        a, b = b, c
    elapsed = time.perf_counter() - start
    estimated_flops = repeats * 2 * (matrix_size ** 3)
    return {
        "cpu_matrix_size": matrix_size,
        "cpu_repeats": repeats,
        "cpu_elapsed_s": elapsed,
        "cpu_gflops": estimated_flops / elapsed / 1e9,
        "cpu_checksum": checksum,
    }


def memory_kernel(array_gb, cycles, seed):
    rng = np.random.default_rng(seed)
    target_bytes = int(array_gb * (1024 ** 3))
    element_size = np.dtype(np.float64).itemsize
    element_count = max(1, target_bytes // element_size)
    work = rng.random(element_count)
    checksum = 0.0
    start = time.perf_counter()
    for _ in range(cycles):
        np.multiply(work, 1.000001, out=work)
        np.add(work, 3.14159, out=work)
        np.sqrt(work, out=work)
        checksum += float(work[:: max(1, element_count // 10000)].sum())
    elapsed = time.perf_counter() - start
    estimated_bytes = work.nbytes * 3 * cycles
    return {
        "memory_array_gb": array_gb,
        "memory_cycles": cycles,
        "memory_elapsed_s": elapsed,
        "memory_est_bandwidth_gb_s": estimated_bytes / elapsed / (1024 ** 3),
        "memory_checksum": checksum,
    }


def cpu_workload(config):
    return cpu_kernel(
        matrix_size=config["cpu_matrix_size"],
        repeats=config["cpu_repeats"],
        seed=config["random_seed"],
    )


def memory_workload(config):
    return memory_kernel(
        array_gb=config["memory_array_gb"],
        cycles=config["memory_cycles"],
        seed=config["random_seed"],
    )


def disk_write_workload(config):
    target_path = ARTIFACT_DIR / f"disk_write_{RUN_ID}.bin"
    metrics = write_file_benchmark(
        target_path=target_path,
        file_size_mb=config["disk_file_size_mb"],
        chunk_mb=config["disk_chunk_mb"],
    )
    if config["cleanup_temp_files"]:
        safe_unlink(target_path)
    return metrics


def disk_read_workload(config):
    seed_path = ARTIFACT_DIR / f"disk_read_seed_{config['disk_file_size_mb']}mb.bin"
    ensure_seed_file(
        seed_path=seed_path,
        file_size_mb=config["disk_file_size_mb"],
        chunk_mb=config["disk_chunk_mb"],
    )
    return read_file_benchmark(source_path=seed_path, chunk_mb=config["disk_chunk_mb"])


def combined_workload(config):
    read_seed_path = ARTIFACT_DIR / f"combined_read_seed_{config['disk_file_size_mb']}mb.bin"
    ensure_seed_file(
        seed_path=read_seed_path,
        file_size_mb=config["disk_file_size_mb"],
        chunk_mb=config["disk_chunk_mb"],
    )
    write_target_path = ARTIFACT_DIR / f"combined_write_{RUN_ID}.bin"

    wall_start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=4) as executor:
        future_map = {
            "cpu": executor.submit(
                cpu_kernel,
                config["combined_cpu_matrix_size"],
                config["combined_cpu_repeats"],
                config["random_seed"] + 1,
            ),
            "memory": executor.submit(
                memory_kernel,
                config["combined_memory_gb"],
                config["combined_memory_cycles"],
                config["random_seed"] + 2,
            ),
            "write": executor.submit(
                write_file_benchmark,
                write_target_path,
                config["disk_file_size_mb"],
                config["disk_chunk_mb"],
            ),
            "read": executor.submit(
                read_file_benchmark,
                read_seed_path,
                config["disk_chunk_mb"],
            ),
        }
        results = {name: future.result() for name, future in future_map.items()}
    wall_elapsed = time.perf_counter() - wall_start

    if config["cleanup_temp_files"]:
        safe_unlink(write_target_path)

    overlap_factor = (
        results["cpu"]["cpu_elapsed_s"]
        + results["memory"]["memory_elapsed_s"]
        + results["write"]["disk_write_elapsed_s"]
        + results["read"]["disk_read_elapsed_s"]
    ) / wall_elapsed

    return {
        "combined_wall_s": wall_elapsed,
        "combined_overlap_factor": overlap_factor,
        "combined_cpu_gflops": results["cpu"]["cpu_gflops"],
        "combined_memory_bandwidth_gb_s": results["memory"]["memory_est_bandwidth_gb_s"],
        "combined_write_mb_s": results["write"]["disk_write_mb_s"],
        "combined_read_mb_s": results["read"]["disk_read_mb_s"],
        "combined_cpu_elapsed_s": results["cpu"]["cpu_elapsed_s"],
        "combined_memory_elapsed_s": results["memory"]["memory_elapsed_s"],
        "combined_write_elapsed_s": results["write"]["disk_write_elapsed_s"],
        "combined_read_elapsed_s": results["read"]["disk_read_elapsed_s"],
    }

In [ ]:
RUNS = {}
TEST_PLAN = [
    ("cpu_only", cpu_workload),
    ("memory_only", memory_workload),
    ("disk_write_only", disk_write_workload),
    ("disk_read_only", disk_read_workload),
    ("combined_parallel", combined_workload),
]

persist_json(ARTIFACT_DIR / "config.json", CONFIG)
persist_json(ARTIFACT_DIR / "environment.json", ENV_INFO)

for test_name, workload in TEST_PLAN:
    print(f"Running {test_name}...")
    RUNS[test_name] = run_with_monitor(test_name, workload, CONFIG)
    RUNS[test_name]["samples"].to_csv(ARTIFACT_DIR / f"{test_name}_timeseries.csv", index=False)
    persist_json(ARTIFACT_DIR / f"{test_name}_summary.json", RUNS[test_name]["summary"])

SUMMARY_DF = pd.DataFrame([entry["summary"] for entry in RUNS.values()])
summary_order = [name for name, _ in TEST_PLAN]
SUMMARY_DF["test_name"] = pd.Categorical(SUMMARY_DF["test_name"], categories=summary_order, ordered=True)
SUMMARY_DF = SUMMARY_DF.sort_values("test_name").reset_index(drop=True)
SUMMARY_DF.to_csv(ARTIFACT_DIR / "summary.csv", index=False)

display_columns = [
    "test_name",
    "elapsed_s",
    "avg_system_cpu_pct",
    "peak_system_cpu_pct",
    "peak_process_rss_gb",
    "min_available_memory_gb",
    "cpu_gflops",
    "memory_est_bandwidth_gb_s",
    "disk_write_mb_s",
    "disk_read_mb_s",
    "combined_overlap_factor",
    "combined_write_mb_s",
    "combined_read_mb_s",
]
display_columns = [column for column in display_columns if column in SUMMARY_DF.columns]
SUMMARY_DF[display_columns].round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

SUMMARY_DF.plot.bar(x="test_name", y="elapsed_s", ax=axes[0], color="#2A9D8F", legend=False)
axes[0].set_title("Temps d'execution")
axes[0].set_ylabel("secondes")
axes[0].tick_params(axis="x", rotation=25)

SUMMARY_DF.plot.bar(x="test_name", y="peak_process_rss_gb", ax=axes[1], color="#E76F51", legend=False)
axes[1].set_title("Pic memoire RSS")
axes[1].set_ylabel("GB")
axes[1].tick_params(axis="x", rotation=25)

SUMMARY_DF.plot.bar(x="test_name", y="peak_system_cpu_pct", ax=axes[2], color="#264653", legend=False)
axes[2].set_title("Pic CPU systeme")
axes[2].set_ylabel("%")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

throughput_cols = [
    "disk_write_mb_s",
    "disk_read_mb_s",
    "combined_write_mb_s",
    "combined_read_mb_s",
]
throughput_df = SUMMARY_DF[["test_name"] + throughput_cols].melt(
    id_vars="test_name",
    value_vars=throughput_cols,
    var_name="metric",
    value_name="mb_s",
).dropna()

plt.figure(figsize=(12, 5))
for metric_name, group in throughput_df.groupby("metric"):
    plt.bar(group["test_name"].astype(str) + " | " + metric_name, group["mb_s"])
plt.title("Debits disque")
plt.ylabel("MB/s")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
TEST_TO_INSPECT = "combined_parallel"
samples = RUNS[TEST_TO_INSPECT]["samples"].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

axes[0].plot(samples["t_s"], samples["system_cpu_pct"], label="system_cpu_pct", color="#264653")
axes[0].plot(samples["t_s"], samples["process_cpu_pct"], label="process_cpu_pct", color="#2A9D8F")
axes[0].set_ylabel("CPU %")
axes[0].set_title(f"Series temporelles - {TEST_TO_INSPECT}")
axes[0].legend()

axes[1].plot(samples["t_s"], samples["process_rss_gb"], label="process_rss_gb", color="#E76F51")
axes[1].plot(samples["t_s"], samples["available_memory_gb"], label="available_memory_gb", color="#F4A261")
axes[1].set_ylabel("GB")
axes[1].legend()

axes[2].plot(samples["t_s"], samples["system_disk_read_mb"], label="system_disk_read_mb", color="#457B9D")
axes[2].plot(samples["t_s"], samples["system_disk_write_mb"], label="system_disk_write_mb", color="#8D99AE")
axes[2].set_xlabel("temps (s)")
axes[2].set_ylabel("MB cumules")
axes[2].legend()

plt.tight_layout()
plt.show()

## Comment lire les resultats

- `cpu_only` : regarde `cpu_gflops`, `avg_system_cpu_pct` et `peak_system_cpu_pct`.
- `memory_only` : regarde `peak_process_rss_gb`, `min_available_memory_gb` et `memory_est_bandwidth_gb_s`.
- `disk_write_only` et `disk_read_only` : regarde surtout `disk_write_mb_s` et `disk_read_mb_s`.
- `combined_parallel` : compare `combined_write_mb_s`, `combined_read_mb_s` et `combined_overlap_factor` avec les tests isoles.
- Si `combined_overlap_factor` est nettement superieur a `1`, cela veut dire qu'il y a bien eu de la superposition entre les charges.
- Si les debits disque ou les performances CPU chutent fortement en mode combine, tu as probablement un goulot d'etranglement sur le stockage, la RAM, le scheduler CPU, ou le partage de ressources entre utilisateurs.

Pour un benchmark plus proche de la production :
- augmente `disk_file_size_mb` a `2048` ou `4096` si tu as assez d'espace disque
- augmente `memory_array_gb` progressivement
- lance le notebook plusieurs fois et compare les CSV
- execute les tests a des heures creuses pour reduire le bruit de fond